# 🤖 05 — Model Building & Evaluation

**Objective**: Train 5 classification models, compare their performance, select the best one, tune its hyperparameters, and analyze feature importance.

**Why this step matters**: Different algorithms have different strengths. By comparing them systematically with consistent metrics, we can select the model that best balances accuracy, precision, recall, and business utility.

**Models Compared**:
1. **Logistic Regression** — interpretable linear baseline
2. **Decision Tree** — non-linear, easy to visualize
3. **Random Forest** — ensemble, reduces overfitting
4. **XGBoost** — gradient boosting, state-of-the-art for tabular data
5. **LightGBM** — fast gradient boosting, memory efficient

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from src.model_trainer import (
    FEATURE_COLS, TARGET_COL, split_data, get_models,
    evaluate_model, train_and_compare, tune_best_model,
    get_feature_importance, save_model
)
from src.utils import (
    set_plot_style, COLORS, plot_confusion_matrix,
    plot_roc_curves, plot_feature_importance
)

set_plot_style()

# Load ML dataset
df = pd.read_csv('../data/ml_dataset.csv')
print(f"Loaded ML dataset: {df.shape[0]:,} customers × {df.shape[1]} columns")
print(f"Features: {FEATURE_COLS}")
print(f"Target: {TARGET_COL} (churn rate: {df[TARGET_COL].mean()*100:.1f}%)")

## 1. Train-Test Split

In [ ]:
# Stratified 80/20 split
X_train, X_test, y_train, y_test = split_data(df, test_size=0.2, random_state=42)

print(f"\nTrain shape: {X_train.shape}")
print(f"Test shape:  {X_test.shape}")

## 2. Train & Compare All Models

In [ ]:
# Train all 5 models and compare
comparison = train_and_compare(X_train, X_test, y_train, y_test)
comparison

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Grouped bar chart of metrics
metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
x = np.arange(len(comparison))
width = 0.15

for i, metric in enumerate(metrics):
    axes[0].bar(x + i*width, comparison[metric], width, label=metric.replace('_', ' ').title())

axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels(comparison['model'], rotation=30, ha='right', fontsize=9)
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.1)

# ROC-AUC comparison
colors = [COLORS['primary'], COLORS['secondary'], COLORS['success'],
          COLORS['warning'], COLORS['danger']]
axes[1].barh(comparison['model'][::-1], comparison['roc_auc'][::-1],
             color=colors[:len(comparison)][::-1], edgecolor='none', height=0.5)
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_title('ROC-AUC Comparison', fontweight='bold', fontsize=14)
for i, v in enumerate(comparison['roc_auc'][::-1]):
    axes[1].text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## 3. ROC Curve Comparison

In [ ]:
# Plot ROC curves for all models
models = get_models()
trained_models = {}
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        trained_models[name] = model
    else:
        model.fit(X_train, y_train)
        trained_models[name] = model

# Plot ROC curves (using original X_test for tree-based, scaled for LR)
fig = plot_roc_curves(
    {k: v for k, v in trained_models.items() if k != "Logistic Regression"},
    X_test, y_test
)
plt.show()

## 4. Select Best Model

Based on the comparison above, we select the model with the **highest ROC-AUC** score. ROC-AUC is our primary metric because:
- It's **threshold-independent** — measures ranking quality
- It handles **class imbalance** better than accuracy
- It captures both precision and recall trade-offs

In [ ]:
# Select best model
best_model_name = comparison.iloc[0]['model']
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   ROC-AUC: {comparison.iloc[0]['roc_auc']:.4f}")
print(f"   F1-Score: {comparison.iloc[0]['f1_score']:.4f}")

## 5. Hyperparameter Tuning

In [ ]:
# Tune the best model with RandomizedSearchCV
tuned_model = tune_best_model(X_train, y_train, model_name=best_model_name, n_iter=30)

# Evaluate tuned model
tuned_metrics = evaluate_model(tuned_model, X_test, y_test)
print(f"\nTuned Model Performance:")
for metric, value in tuned_metrics.items():
    print(f"  {metric:12s}: {value:.4f}")

## 6. Detailed Evaluation of Tuned Model

In [ ]:
# Classification report
y_pred = tuned_model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

In [ ]:
# Confusion matrix
fig = plot_confusion_matrix(y_test, y_pred, title=f'{best_model_name} — Confusion Matrix')
plt.show()

# Interpretation
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"\nConfusion Matrix Interpretation:")
print(f"  True Negatives (correctly predicted Active):  {tn}")
print(f"  True Positives (correctly predicted Churned): {tp}")
print(f"  False Positives (Active predicted as Churned): {fp}")
print(f"  False Negatives (Churned predicted as Active): {fn} ← most costly error")

## 7. Feature Importance

In [ ]:
# Feature importance analysis
fi = get_feature_importance(tuned_model, FEATURE_COLS)
print("Feature Importance Ranking:")
print(fi.to_string(index=False))

fig = plot_feature_importance(fi, title=f'{best_model_name} — Feature Importance')
plt.show()

## 8. Save the Best Model

In [ ]:
# Save for deployment in Streamlit
save_model(tuned_model, 'best_model.pkl')

# Also save the model comparison results
comparison.to_csv('../data/model_comparison.csv', index=False)
fi.to_csv('../data/feature_importance.csv', index=False)
print("\n✓ All artifacts saved!")

## Summary

| Aspect | Result |
|--------|--------|
| Best Model | (determined by comparison — likely XGBoost or LightGBM) |
| Tuning Method | RandomizedSearchCV (30 iterations, 5-fold CV) |
| Primary Metric | ROC-AUC |
| Top Feature | `recency_days` (days since last purchase) |

**Why this model?**: Tree-based gradient boosting models (XGBoost, LightGBM) consistently outperform on tabular data because they:
- Handle non-linear relationships naturally
- Don't require feature scaling
- Have built-in feature importance
- Handle class imbalance with `scale_pos_weight` / `class_weight`

**Next Step**: In notebook 06, we'll translate these results into actionable business insights.